In [ ]:
# ── Hero dashboard (code hidden; click "Show hidden code" to read it) ─────────────────────────────
import csv, json, os, sys, glob, math, time, collections, subprocess, shutil
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
csv.field_size_limit(10**9)
INPUT = Path("/kaggle/input") if Path("/kaggle/input").is_dir() else Path("input")
def find(name, must=True):
    hits = sorted(INPUT.rglob(name))
    if not hits and must: raise FileNotFoundError(name)
    return hits[0] if hits else None
DS = find("our_programs.csv").parent              # our dataset
class Puzzle:
    __slots__ = ("id", "starts", "targets", "limit", "registers")
    def __init__(s, *a): s.id, s.starts, s.targets, s.limit, s.registers = a
def load_puzzles(path):
    with open(path, newline="", encoding="utf-8-sig") as f:
        return {r["id"]: Puzzle(r["id"], json.loads(r["starting_values"]), json.loads(r["targets"]),
                                int(r["value_limit"]), int(r["registers"])) for r in csv.DictReader(f)}
def read_submission(path):
    with open(path, newline="", encoding="utf-8-sig") as f:
        rd = csv.reader(f); assert next(rd) == ["id", "program"]
        return {pid: json.loads(prog) for pid, prog in rd}
PUZ = load_puzzles(find("puzzles.csv"))
OURS = read_submission(DS / "our_programs.csv"); FRONTIER = read_submission(DS / "public_frontier.csv")
FAM = pd.read_csv(DS / "families.csv").set_index("id").fam.to_dict()
LADDER = json.load(open(DS / "ladder.json"))
HIST = pd.read_csv(DS / "history.csv")

# """Hero dashboard for the notebook: where our instructions were saved, and what a rewrite looks like."""
import math, json, collections
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

# reference palette (validated light-mode categorical order) + text/surface tokens
SURF, INK, INK2, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#8a8984", "#e6e5e0"
BLUE, ORANGE, AQUA, YELLOW, MAGENTA, GREEN, VIOLET, RED = (
    "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948")
SEQ = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]

def style():
    plt.rcParams.update({
        "figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF,
        "axes.edgecolor": GRID, "axes.labelcolor": INK2, "axes.titlecolor": INK,
        "axes.titleweight": "bold", "axes.titlesize": 12.5, "axes.titlelocation": "left",
        "axes.labelsize": 10, "xtick.color": INK2, "ytick.color": INK2,
        "xtick.labelsize": 9, "ytick.labelsize": 9, "text.color": INK,
        "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.7,
        "axes.spines.top": False, "axes.spines.right": False, "axes.spines.left": False,
        "legend.frameon": False, "legend.fontsize": 9, "font.family": "DejaVu Sans",
        "axes.axisbelow": True, "figure.dpi": 110,
    })

OPF = {'+': lambda a, b: a + b, '-': lambda a, b: a - b, '*': lambda a, b: a * b, '/': lambda a, b: a // b}

def dag_of(p, prog):
    """value -> (op, a, b, step) for every value the program computes and a target depends on."""
    regs = list(p.starts) + [None] * (p.registers - len(p.starts))
    nodes = {v: (None, None, None, -1) for v in p.starts}
    for k, (op, i, j, d) in enumerate(prog):
        a, b = regs[i], regs[j]; v = OPF[op](a, b); regs[d] = v
        nodes.setdefault(v, (op, a, b, k))
    need, st = set(), list(set(p.targets))
    while st:
        x = st.pop()
        if x in need: continue
        need.add(x); op, a, b, _ = nodes[x]
        if op: st += [a, b]
    return {v: nodes[v] for v in need | set(p.starts)}

def layered_positions(dag):
    depth = {}
    for v in sorted(dag, key=lambda v: dag[v][3]):
        op, a, b, _ = dag[v]
        depth[v] = 0 if op is None else 1 + max(depth[a], depth[b])
    by = collections.defaultdict(list)
    for v, d in depth.items(): by[d].append(v)
    pos = {}
    for d, vs in by.items():
        vs.sort(key=lambda v: dag[v][3])
        for i, v in enumerate(vs): pos[v] = (d, (i + 0.5) / len(vs))
    return pos, depth

def draw_dag(ax, p, dag, highlight=frozenset(), title="", hl_color=ORANGE):
    pos, depth = layered_positions(dag)
    T = set(p.targets)
    for v, (op, a, b, _) in dag.items():
        if op is None: continue
        for u in {a, b}:
            x0, y0 = pos[u]; x1, y1 = pos[v]
            new = v in highlight
            ax.plot([x0, x1], [y0, y1], color=hl_color if new else "#c9c8c2", lw=1.1 if new else 0.5,
                    alpha=0.9 if new else 0.55, zorder=1 if not new else 2)
    for v in dag:
        x, y = pos[v]; op = dag[v][0]
        if op is None: c, s, m = INK, 46, "s"
        elif v in T: c, s, m = (hl_color if v in highlight else BLUE), 44, "o"
        else: c, s, m = (hl_color if v in highlight else "#b9b8b2"), 16, "o"
        ax.scatter([x], [y], s=s, c=c, marker=m, zorder=3, edgecolors=SURF, linewidths=1.2)
    ax.set_title(title, fontsize=11)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    for sp in ax.spines.values(): sp.set_visible(False)
    ax.set_xlabel("arithmetic depth from the starting values  →", fontsize=9, color=INK2)

def _comma(ax, axis="y"):
    from matplotlib.ticker import FuncFormatter
    f = FuncFormatter(lambda v, _: f"{v:,.0f}")
    (ax.yaxis if axis == "y" else ax.xaxis).set_major_formatter(f)

def dashboard(P, base, ours, fam, ladder, history, public_hist, showcase, out_png=None, score_label="ours", lb=None):
    """P: puzzles; base: public frontier programs; ours: our programs; fam: {pid: family};
    ladder: [(name, total)] public outputs; history: [(minutes, total)] ours;
    public_hist: [(minutes, total)] public frontier; showcase: pid for the rewrite anatomy."""
    style()
    fig = plt.figure(figsize=(17, 17.5))
    gs = fig.add_gridspec(3, 6, height_ratios=[1.0, 1.0, 1.15], hspace=0.42, wspace=0.6, top=0.875, bottom=0.04)
    b_tot = sum(len(base[k]) for k in P); o_tot = sum(len(ours[k]) for k in P)
    gain = {k: len(base[k]) - len(ours[k]) for k in P}
    n_imp = sum(1 for g in gain.values() if g > 0)
    n_t = sum(len(set(p.targets) - set(p.starts)) for p in P.values())
    fig.text(0.125, 0.985, "Santa 2026 \u00b7 Elf Calculator \u2014 a value-DAG search that shortens the public frontier",
             fontsize=18, fontweight="bold", va="top")
    tiles = [(f"{lb if lb else o_tot:,}", "Kaggle LB score (total instructions)" if lb else "total instructions"),
             (f"\u2212{b_tot - o_tot:,}", f"vs best public merge ({b_tot:,})"),
             (f"{n_imp}", "puzzles beat every public program"),
             (f"{o_tot / n_t:.3f}", f"instructions per target ({n_t:,} targets)")]
    for i, (big, small) in enumerate(tiles):
        x = 0.125 + i * 0.205
        fig.text(x, 0.955, big, fontsize=30, fontweight="bold", color=BLUE if i == 1 else INK, va="top")
        fig.text(x, 0.912, small, fontsize=10.5, color=INK2, va="top")

    # score ladder
    ax = fig.add_subplot(gs[0, :3])
    lad = sorted(ladder, key=lambda x: x[1])[:9]
    names = [n for n, _ in lad] + [score_label]; vals = [v for _, v in lad] + [o_tot]
    order = np.argsort(vals)[::-1]; names = [names[i] for i in order]; vals = [vals[i] for i in order]
    cols = [BLUE if n == score_label else "#b9cde6" for n in names]
    y = np.arange(len(vals))
    ax.barh(y, vals, color=cols, height=0.7)
    lo = min(vals) - 0.35 * (max(vals) - min(vals) + 100); hi = max(vals) + 0.22 * (max(vals) - lo)
    ax.set_xlim(lo, hi)
    ax.set_yticks(y); ax.set_yticklabels([n if len(n) < 42 else n[:40] + "\u2026" for n in names], fontsize=8.6)
    for yi, v, n in zip(y, vals, names):
        ax.text(v + 0.01 * (hi - lo), yi, f"{v:,}", va="center", fontsize=8.8,
                color=INK if n == score_label else INK2, fontweight="bold" if n == score_label else None)
    ax.set_title("Every public output we replayed, and ours")
    ax.set_xlabel("total instructions over 1,000 puzzles (lower is better)"); ax.grid(axis="y", visible=False); _comma(ax, "x")

    # progress
    ax = fig.add_subplot(gs[0, 3:])
    if public_hist:
        px = [h[0] for h in public_hist] + [max(h[0] for h in history + public_hist)]
        py = [h[1] for h in public_hist] + [public_hist[-1][1]]
        ax.step(px, py, where="post", color=MUTED, lw=1.6, ls="--")
        ax.text(px[0], py[0], "  best public notebook", va="bottom", ha="left", fontsize=9, color=INK2)
    if history:
        hx = [h[0] for h in history]; hy = [h[1] for h in history]
        ax.step(hx, hy, where="post", color=BLUE, lw=2.2)
        ax.scatter(hx, hy, color=BLUE, s=14, zorder=3)
        ax.text(hx[-1], hy[-1], f" {hy[-1]:,}", va="top", ha="right", fontsize=10, fontweight="bold", color=INK)
    ax.set_title("Score over search time (20 CPU cores)")
    ax.set_xlabel("minutes since we started"); ax.set_ylabel("total instructions"); _comma(ax)

    # savings map
    ax = fig.add_subplot(gs[1, :4])
    ids = sorted(P); W = 50
    grid = np.full((len(ids) // W, W), np.nan)
    for i, k in enumerate(ids): grid[i // W, i % W] = gain[k]
    from matplotlib.colors import ListedColormap, BoundaryNorm
    gmax = max(1, int(np.nanmax(grid)))
    cm = ListedColormap(["#efeeea"] + SEQ[2:][:min(5, gmax)])
    bounds = [-0.5, 0.5] + [x + 0.5 for x in range(1, min(5, gmax) + 1)]
    if gmax > 5: bounds[-1] = gmax + 0.5
    ax.imshow(grid, cmap=cm, norm=BoundaryNorm(bounds, cm.N), aspect="auto", interpolation="nearest")
    ax.set_xticks([]); ax.set_yticks(range(0, len(ids) // W, 5)); ax.set_yticklabels([f"{r * W + 1:04d}" for r in range(0, len(ids) // W, 5)], fontsize=8)
    ax.grid(False)
    ax.set_title("The 1,000 puzzles, one cell each \u2014 colour = instructions we removed")
    hand = [mpatches.Patch(color="#efeeea", label="unchanged")] + [mpatches.Patch(color=cm(i + 1), label=(f"\u2212{i + 1}" if i + 1 < min(5, gmax) or gmax <= 5 else f"\u2212{i + 1}+")) for i in range(min(5, gmax))]
    ax.legend(handles=hand, loc="upper left", bbox_to_anchor=(0, -0.02), ncol=7, fontsize=9)
    ax.set_ylabel("puzzle id (rows of 50)")

    # by family
    ax = fig.add_subplot(gs[1, 4:])
    fams = ["independent", "mixed", "shared", "multiplicative", "tiny"]
    cnt = [sum(1 for k in P if fam[k] == f) for f in fams]
    sv = [sum(gain[k] for k in P if fam[k] == f) for f in fams]
    imp = [sum(1 for k in P if fam[k] == f and gain[k] > 0) for f in fams]
    ax.barh(range(len(fams))[::-1], sv, color=BLUE, height=0.6)
    for i, (v, c, m) in enumerate(zip(sv, cnt, imp)):
        ax.text(v, len(fams) - 1 - i, f"  \u2212{v:,}  ({m}/{c} puzzles)", va="center", fontsize=9, color=INK2)
    ax.set_yticks(range(len(fams))[::-1]); ax.set_yticklabels(fams)
    ax.set_xlim(0, max(sv + [1]) * 1.9)
    ax.set_title("Instructions removed, by puzzle family*"); ax.grid(axis="y", visible=False)

    # rewrite anatomy
    p = P[showcase]; d0, d1 = dag_of(p, base[showcase]), dag_of(p, ours[showcase])
    ax = fig.add_subplot(gs[2, :3])
    rewrite_panel(ax, p, d0, d1, "old", f"{showcase} \u00b7 public program: the part we removed ({len(base[showcase])} instr.)")
    ax = fig.add_subplot(gs[2, 3:])
    rewrite_panel(ax, p, d0, d1, "new", f"{showcase} \u00b7 our replacement ({len(ours[showcase])} instr.)")
    handles = [mpatches.Patch(color="#eeede8", label="value kept from the program"),
               mpatches.Patch(color=BLUE, label="target"),
               mpatches.Patch(color=RED, label="removed instruction (op \u2192 value)"),
               mpatches.Patch(color=AQUA, label="new instruction (op \u2192 value)")]
    ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(-0.1, -0.02), ncol=4, fontsize=9.5)
    fig.text(0.0, 0.005, "*family inferred from practice-set fingerprints (start-value size, target relations). "
             "Every program counted here was replayed by an exact referee.", fontsize=8.5, color=INK2)
    if out_png: fig.savefig(out_png, bbox_inches="tight", dpi=110)
    return fig

def fmt_val(v):
    return f"{v:,}"

def rewrite_panel(ax, p, dag_old, dag_new, which, title):
    """Draw only the part of the program that changed: nodes unique to `which` side, their operands
    (context) and the targets they feed. Each node is labelled with its value and recipe."""
    d = dag_old if which == "old" else dag_new
    other = dag_new if which == "old" else dag_old
    changed = [v for v in d if v not in other]
    T = set(p.targets)
    ctx = set()
    for v in changed:
        op, a, b, _ = d[v]
        if op: ctx.update(x for x in (a, b) if x not in changed)
    consumers = set()
    for v, (op, a, b, _) in d.items():
        if op and (a in changed or b in changed) and v not in changed:
            consumers.add(v); ctx.update(x for x in (a, b) if x not in changed)
    nodes = list(dict.fromkeys(list(ctx) + changed + list(consumers)))
    if not changed:
        ax.text(0.5, 0.5, "same program", ha="center", va="center", color=INK2); ax.axis("off"); ax.set_title(title, fontsize=11.5); return
    # depth inside the sub-graph
    dep = {}
    for v in sorted(nodes, key=lambda v: (d[v][3], v)):
        op, a, b, _ = d[v]
        if (v in ctx and v not in consumers) or op is None: dep[v] = 0
        else: dep[v] = 1 + max(dep.get(a, 0), dep.get(b, 0))
    cols = collections.defaultdict(list)
    for v in nodes: cols[dep[v]].append(v)
    pos = {}
    for c, vs in cols.items():
        vs.sort(key=lambda v: d[v][3])
        for i, v in enumerate(vs): pos[v] = (c, 1 - (i + 0.5) / max(len(vs), 1))
    hl = RED if which == "old" else AQUA
    for v in nodes:
        op, a, b, _ = d[v]
        if not op or (v in ctx and v not in consumers): continue
        for u in {a, b}:
            if u in pos:
                ax.annotate("", xy=pos[v], xytext=pos[u],
                            arrowprops=dict(arrowstyle="-|>", color=hl if v in changed else "#b9b8b2", lw=1.2,
                                            shrinkA=9, shrinkB=9, mutation_scale=9))
    for v in nodes:
        x, y = pos[v]; op, a, b, _ = d[v]
        if v in changed: fc, ec, tc = hl, hl, "white"
        elif v in T: fc, ec, tc = BLUE, BLUE, "white"
        else: fc, ec, tc = "#eeede8", "#c9c8c2", INK2
        label = fmt_val(v)
        if (v in changed or v in consumers) and op: label = f"{op} → {label}"
        ax.text(x, y, label, ha="center", va="center", fontsize=7.6, color=tc, zorder=4,
                bbox=dict(boxstyle="round,pad=0.32", fc=fc, ec=ec, lw=1))
    ax.set_xlim(-0.6, max(dep.values()) + 0.6 if dep else 1); ax.set_ylim(-0.05, 1.05)
    ax.axis("off"); ax.set_title(title, fontsize=11.5)


ours_h = [(r.minutes, r.total) for r in HIST.itertuples() if r.label != "public_best"]
ours_h = [(0.0, int(HIST.total.iloc[0]))] + ours_h
pub_h = [(r.minutes, r.total) for r in pd.read_csv(DS / "public_history.csv").itertuples()]
gain = {k: len(FRONTIER[k]) - len(OURS[k]) for k in PUZ}
SHOW = sorted([k for k in PUZ if gain[k] >= 1],
              key=lambda k: len(dag_of(PUZ[k], FRONTIER[k]).keys() ^ dag_of(PUZ[k], OURS[k]).keys()))
SHOW = SHOW or [min(PUZ, key=lambda k: len(OURS[k]))]
fig = dashboard(PUZ, FRONTIER, OURS, FAM, [tuple(x) for x in LADDER], ours_h, pub_h, SHOW[0], lb=132256)
plt.show()

# Santa 2026 | LNS + CP-SAT Recombine | LB 132,256

**Leaderboard score: 132,256 instructions** — 1 fewer than the best public notebook output at the time of our run
(132,257, the per-puzzle best of every public output listed below). 1 of the 1,000 puzzles are shorter than anything public.

The figure above is the whole story in one screen: the score ladder, our score over search time, a map of the 1,000 puzzles
coloured by how many instructions we removed, which puzzle families gained, and one real rewrite (the instructions we deleted
from a public program, in red, and what replaced them, in green, with every value printed).

### Version history (newest first)

| Version | Date (KST) | Our LB | Best public notebook then | What changed |
|---:|---|---:|---:|---|
| v90 | 2026-11-22 07:31 | **132,256** | 132,257 | Re-merged the public frontier (132,257); added 1 of our shorter programs (−1) |
| v89 | 2026-11-22 07:25 | **132,258** | 132,258 | Re-merged the public frontier (132,258); added 0 of our shorter programs (−0) |
| v88 | 2026-11-22 07:15 | **132,262** | 132,262 | Re-merged the public frontier (132,262); added 0 of our shorter programs (−0) |
| v87 | 2026-11-22 07:05 | **132,266** | 132,266 | Re-merged the public frontier (132,266); added 0 of our shorter programs (−0) |
| v86 | 2026-11-22 06:53 | **132,270** | 132,270 | Re-merged the public frontier (132,270); added 0 of our shorter programs (−0) |
| v85 | 2026-11-22 06:43 | **132,274** | 132,274 | Re-merged the public frontier (132,274); added 0 of our shorter programs (−0) |
| v84 | 2026-11-22 06:31 | **132,277** | 132,277 | Re-merged the public frontier (132,277); added 0 of our shorter programs (−0) |
| v83 | 2026-11-22 05:58 | **132,293** | 132,293 | Re-merged the public frontier (132,293); added 0 of our shorter programs (−0) |

## What we reused — thank you 🙏

We did **not** start from scratch. Every program we optimise was first found by other Kagglers, and our engine follows a
public design. Precisely:

| Source | Author | What we used |
|---|---|---|
| [Santa 2026 · 🎄 Best Public Score, a Gift per Run](https://www.kaggle.com/code/asteriospanoras/santa-2026-best-public-score-a-gift-per-run) | Asterios Panoras | output programs — holds the public per-puzzle minimum on 1,000 of 1,000 puzzles of the frontier we merged last |
| [Santa 2026 Reproducible Dictionary Beam Solver](https://www.kaggle.com/code/arsen123/santa-2026-reproducible-dictionary-beam-solver) | Arsen Zakaryan | output programs — holds the public per-puzzle minimum on 999 of 1,000 puzzles of the frontier we merged last |
| [Santa 2026 · ⭐ North Star, Verified Best Score](https://www.kaggle.com/code/asteriospanoras/santa-2026-north-star-verified-best-score) | Asterios Panoras | output programs — holds the public per-puzzle minimum on 999 of 1,000 puzzles of the frontier we merged last |
| [🎄 132,259 · Best Public Score · Santa 2026](https://www.kaggle.com/code/pardheev/132-259-best-public-score-santa-2026) | Pardheev Krishna | output programs — holds the public per-puzzle minimum on 998 of 1,000 puzzles of the frontier we merged last |
| [🎄 132,354 · Best Public Score · Santa 2026](https://www.kaggle.com/code/pardheev/132-354-best-public-score-santa-2026) | Pardheev Krishna | output programs — holds the public per-puzzle minimum on 997 of 1,000 puzzles of the frontier we merged last |
| [Arithmetic Recipe Pool and MILP](https://www.kaggle.com/code/wangjakie/santa-2026-arithmetic-recipe-pool-and-milp) | Wang Jakie | **search engine.** We run its public C++ graph-search strategies (`anneal`, `incremental`, `compact_rotate`, `quartet`) unchanged on every core, with fresh seeds and more rounds; the `*_seeded` variants are our two-line edit that puts near-length historical programs into its crossover pool |
| [IceFire Strong Start](https://www.kaggle.com/code/romanrozen/santa-2026-icefire-strong-start) | Roman Rozen | output programs — the main source of our first merge (999/1,000 puzzles) |
| [137,792 CP-SAT + Multi-Bank Fuser](https://www.kaggle.com/code/santiagozazo/santa-2026-137-792-cp-sat-multi-bank-fuser) | Santiago Zazo | output programs (tie-holder in an earlier frontier) |
| [Elf Calculator – Verified Portfolio](https://www.kaggle.com/code/hanifnoerrofiq/santa-2026-elf-calculator-verified-portfolio) | Hanif Noer Rofiq | output programs — with the Starter Notebook, the 136,829 frontier our very first sweep started from |
| [Public Frontier · seeded graph search](https://www.kaggle.com/code/royalacecat/santa-2026-public-frontier-score-134-459) | E-Max AI (royalacecat) | the idea of seeding Wang Jakie's crossover with valid near-length historical programs, and the donor-retaining `*_diverse` crossover pool we now run unchanged from the public dataset [Retained Graph Search Source](https://www.kaggle.com/datasets/royalacecat/santa-2026-retained-graph-search-source) |
| [Joint Recipes for Large Puzzles](https://www.kaggle.com/code/atakanaldemir/santa-2026-joint-recipes-for-large-puzzles) | Atakan Aldemir | **optimizers.** We run its public C++ multiplier-frontier and interface-closure optimizers unchanged (our driver only feeds them our programs and replays every result); source and NOTICE in the dataset |
| [Santa_2026 (GitHub)](https://github.com/Hansen0223/Santa_2026) | Hansen0223 | programs from its public-assisted checkpoint and its independent-solver candidate (merged per puzzle; its README credits the Apache-2.0 upstream work) |
| [Santa is not happy](https://www.kaggle.com/code/prvsiyan/santa-is-not-happy) | prvsiyan | **engine design.** Our C++ engine re-implements the value-DAG large-neighbourhood search of its `lns.cpp` (Apache-2.0): the value DAG, the inverse-operation table, “remove a target's exclusive cone → rebuild with an exact search over all remaining values → re-schedule into 48 registers”. We read that code and wrote our own implementation; the structure is theirs. |
| [Elf Calculator With Register Reuse](https://www.kaggle.com/code/tonylica/elf-calculator-with-register-reuse) / [Wider Dictionary + Portfolio](https://www.kaggle.com/code/konbu17/santa-2026-wider-dictionary-portfolio-163k) | Tony Li / konbu17 | the backward dictionary search that produced the programs everyone (including us) builds on |
| [Elf Calculator, explained from zero](https://www.kaggle.com/code/sheroleg/santa-2026-elf-calculator-explained-from-zero), [Inside the Elf's Calculator](https://www.kaggle.com/code/evgendvorkin/santa-2026-inside-the-elf-s-calculator), [Best public score + dataset + fails](https://www.kaggle.com/code/asteriospanoras/santa-2026-best-public-score-dataset-fails) | Oleg Sher, Дворкин Евгений, Asterios Panoras | referee conventions, family analysis and the list of ideas that did not work, which saved us time |

The public programs are themselves merges of earlier work: IceFire credits Tony Li, konbu17, lightningv08, Atakan Aldemir,
Aman Atar, Elias T.-H. Shieh, Ángel Jacinto Sánchez Ruiz, Georgy Mamarin, Bikram Basnet, Hanif Noer Rofiq, Nilanjan Saha and
lucifer19. Thank you all — please upvote their notebooks too.

*Note on the moving frontier.* After version 1 of this notebook, the public frontier notebooks merged our released programs
(that is how the frontier moved from 136,669 to 132,257). This version re-merges their newer outputs, so the programs above contain
work from both directions; the 1 puzzles counted as ours are those where our program is **strictly shorter** than every public output.

## What we added

1. **A depth-5 *near-miss* product search.** Exact hashing only finds `T = X op Y` when both sides already exist. We also look for
   `T = X·Y ± R` where `Y` is taken from the **sorted neighbours of T/X** in the sorted array of (pool ∪ one-step values), so the
   product only has to land *close* to `T`, and the residual `R` is cheap. Three shapes: `X,Y,R` each ≤ 1 step; `Y = ⌊T/X⌋` built in
   two steps; `R` built in two steps. These are 5-instruction builds the depth-≤4 search cannot see.
2. **Helper-rooted destroy moves.** Besides a target's exclusive cone we also rebuild a *shared helper* together with the sub-tree it
   dominates (nodes whose every path to a target goes through it).
3. **Multi-order repair and a joint pair search** for 2–3-target neighbourhoods (one shared new helper serving two targets).
4. **Plateau walking at scale, with kicks.** Equal-length rewrites are accepted with probability 0.3; after 200 plateau moves without
   progress the search applies an *iterated-local-search kick* (rebuild a random target cone allowing it to grow by up to 2
   instructions) and continues from there, always keeping the best program. In a 60-puzzle A/B test kicks found 2.5× the savings of
   plain plateau walking (5 vs 2 instructions at 15 s per puzzle). Every candidate is re-scheduled into ≤ 48 registers before it is
   kept; 20 CPU cores sweep all 1,000 puzzles round after round.
5. **CP-SAT recombination of harvested recipes.** While the search walks the plateau it records every recipe it accepts
   (`v = a op b`). Per puzzle we pool those recipes with the current program, add every cross recipe `a op b = v` between pooled
   values, and let CP-SAT choose the smallest set of values (one recipe each, acyclic via integer levels) that still builds every
   target. This finds combinations no single visited program contains: our first three passes removed **45, 28 and 4** instructions on
   top of the LNS. Public notebooks recombine *public banks*; we recombine the *search trajectory*.
6. **Orchestrating several public engines on 20 cores.** Since the frontier started moving quickly, we also run Wang Jakie's
   public graph-search strategies at scale (all credited above) and feed everything — their results, our LNS, the CP-SAT
   recombination and every attached public program — into one per-puzzle best file. In the first hour this added
   anneal −23, incremental −16 and compact_rotate −11 instructions on our own frontier. Since 21 Nov we also run Atakan Aldemir's
   multiplier-frontier and interface-closure optimizers and royalacecat's donor-retaining crossover pool (credited above). Our own
   addition here is a *plateau archive*: every equal-length alternative any engine returns is kept and later used as a new
   starting point, so the searches stop restarting from the same converged program.
7. **Measurements.** The public LNS, re-run on the frontier, finds **0** shorter programs (100 puzzles × 15 s); our first 25-minute
   sweep found **134** shorter puzzles (−319). The section *What did not work* lists our negative results.

Everything below is reproducible: the engine source is in the attached dataset, it is compiled and run live in section 4, and every
program is replayed by the referee in section 1 before it is counted.

## 1 · The calculator and an exact referee

48 registers; the starting values sit in registers `0..k-1`. An instruction `["op", i, j, d]` writes `R[i] op R[j]` into `R[d]`,
`op ∈ {+, −, ×, ÷}`. Every result must be a positive integer `≤ value_limit`, division must be exact, and every target must appear
in some register at least once. **Score = total number of instructions over 1,000 puzzles (lower is better).**
One invalid program rejects the whole file, so nothing below is trusted until this referee replays it.

In [ ]:
def replay(p, prog):
    """Length of a valid program; raises ValueError otherwise (official rules)."""
    if len(prog) > 4096: raise ValueError("more than 4096 instructions")
    R = p.registers; regs = list(p.starts) + [None] * (R - len(p.starts))
    need = set(p.targets) - set(p.starts); L = p.limit
    for k, ins in enumerate(prog):
        if not (isinstance(ins, list) and len(ins) == 4): raise ValueError(f"{k}: malformed")
        op, i, j, d = ins
        for r in (i, j, d):
            if type(r) is not int or not 0 <= r < R: raise ValueError(f"{k}: bad register")
        a, b = regs[i], regs[j]
        if a is None or b is None: raise ValueError(f"{k}: empty register")
        if op == "+": v = a + b
        elif op == "-": v = a - b
        elif op == "*": v = a * b
        elif op == "/":
            if a % b: raise ValueError(f"{k}: inexact division")
            v = a // b
        else: raise ValueError(f"{k}: operator {op!r}")
        if not 1 <= v <= L: raise ValueError(f"{k}: value out of range")
        regs[d] = v; need.discard(v)
    if need: raise ValueError(f"{len(need)} targets never produced")
    return len(prog)

# the referee must reject broken programs before we trust it
toy = Puzzle("toy", [3], [1, 2, 5, 8], 9, 48)
assert replay(toy, [["/", 0, 0, 1], ["-", 0, 1, 2], ["+", 0, 2, 3], ["+", 0, 3, 4]]) == 4
for bad in ([["-", 0, 0, 1]], [["*", 0, 0, 1], ["*", 1, 1, 1]], [["+", 2, 2, 1]], [], [["/", 0, 0, 48]]):
    try: replay(toy, bad); raise AssertionError(bad)
    except ValueError: pass
print("referee self-test passed")

## 2 · Inputs: the public frontier, re-scored live, and our programs

Every attached CSV with the header `id,program` is replayed program by program; the shortest valid program per puzzle is kept.
`our_programs.csv` (attached dataset) holds our optimised programs; `public_frontier.csv` is the frontier exactly as it stood when
we ran, kept for the figures. If the attached public notebooks have improved since, the live merge picks that up automatically.

In [ ]:
banks = {}
for path in sorted(INPUT.rglob("*.csv")):
    if path.name in ("puzzles.csv", "practice_puzzles.csv", "sample_submission.csv", "families.csv"): continue
    try: S = read_submission(path)
    except Exception: continue
    name = path.stem if path.parent == DS else path.parent.name
    banks[name] = S
# public banks first, ours last: a tie is credited to the public author, "our_programs" only when strictly shorter
order = sorted(banks, key=lambda n: (n in ("our_programs", "public_frontier"), n == "our_programs", n))
rows, BEST, WHO = [], {}, {}
for name in order:
    S = banks[name]
    ok = tot = 0
    for pid, p in PUZ.items():
        if pid not in S: continue
        try: n = replay(p, S[pid])
        except ValueError: continue
        ok += 1; tot += n
        if pid not in BEST or n < len(BEST[pid]): BEST[pid], WHO[pid] = S[pid], name
    rows.append(dict(source=name, valid_programs=ok, total=tot if ok == len(PUZ) else None))
display(pd.DataFrame(rows).sort_values("total").reset_index(drop=True))
assert len(BEST) == len(PUZ)
print(f"per-puzzle best over all sources: {sum(len(v) for v in BEST.values()):,} instructions")
print("source of the shortest program per puzzle (ties credited to public notebooks):", collections.Counter(WHO.values()).most_common())

## 3 · Our engine in one picture

A program is a **value DAG**: each distinct value has one recipe `v = a op b`; register numbers are assigned only at the end.
One move of the large-neighbourhood search:

1. **Destroy** — pick a neighbourhood: one target and its *exclusive cone* (instructions used only by it), a group of 2–3 targets
   that share helpers, or a *shared helper* with the sub-tree it dominates. Delete those `k` instructions.
2. **Repair** — rebuild the removed values with **at most `k − 1` instructions** (or `k` for a plateau move), searching over
   *every other value in the DAG* (typically 100–280 values, far more than 48 live registers):
   * depth ≤ 3 exactly by hash inversion (`T = w op x ⇒ w = T − x, T + x, x − T, T / x, T·x, x / T`);
   * depth 4 by streaming two-step values against a hash of “needed” values (meet in the middle);
   * **depth 5 by the near-miss product search** `T = X·Y ± R` with `Y` read from the sorted neighbours of `T/X`.
3. **Schedule** — topologically order the new DAG, check that at most 48 values are ever alive (list scheduling if not), and emit
   registers by linear scan. Keep the move only if the program is shorter (or equal, with probability 0.3).

The figure below shows where a typical program spends its instructions and why deeper search is where the slack is: the cost of a
target grows almost linearly with its size for *independent* puzzles, but stays near 4 for *shared* ones.

In [ ]:
style()
fig, axs = plt.subplots(1, 3, figsize=(17, 4.8))
ax = axs[0]
fam_col = {"independent": BLUE, "mixed": ORANGE, "shared": AQUA}
for f, c in fam_col.items():
    xs = [np.mean([t.bit_length() for t in PUZ[k].targets]) for k in PUZ if FAM[k] == f and len(PUZ[k].targets) >= 8]
    ys = [len(OURS[k]) / len(set(PUZ[k].targets) - set(PUZ[k].starts)) for k in PUZ if FAM[k] == f and len(PUZ[k].targets) >= 8]
    ax.scatter(xs, ys, s=11, color=c, alpha=0.75, label=f, edgecolors="none")
ax.set_title("Instructions per target vs target size"); ax.set_xlabel("mean target size (bits)"); ax.set_ylabel("instructions / target")
ax.legend(loc="upper left")
ax = axs[1]
d = dag_of(PUZ["scored_0001"], OURS["scored_0001"]); tl = sorted(set(PUZ["scored_0001"].targets))
mask = {v: 0 for v in d}
for i, t in enumerate(tl): mask[t] |= 1 << i
for v in sorted(d, key=lambda v: -d[v][3]):
    op, a, b, _ = d[v]
    if op: mask[a] |= mask[v]; mask[b] |= mask[v]
share = collections.Counter(min(bin(mask[v]).count("1"), 8) for v in d if d[v][0])
ax.bar(list(share.keys()), list(share.values()), color=BLUE, width=0.7)
ax.set_xticks(range(1, 9)); ax.set_xticklabels([str(i) if i < 8 else "8+" for i in range(1, 9)])
ax.set_title("scored_0001: how many targets each instruction serves"); ax.set_xlabel("targets depending on the value"); ax.set_ylabel("instructions")
ax.grid(axis="x", visible=False)
ax = axs[2]
lens = np.array([len(OURS[k]) for k in PUZ]); nt = np.array([len(set(PUZ[k].targets) - set(PUZ[k].starts)) for k in PUZ])
ax.scatter(nt + np.random.default_rng(0).uniform(-0.3, 0.3, len(nt)), lens, s=8, color=BLUE, alpha=0.5, edgecolors="none")
ax.plot([0, 32], [0, 32], color=MUTED, lw=1, ls="--"); ax.text(30, 36, "lower bound:\n1 per target", fontsize=8.5, color=INK2, ha="right")
ax.set_title("Program length vs number of targets"); ax.set_xlabel("targets in the puzzle"); ax.set_ylabel("instructions")
plt.tight_layout(); plt.show()

## 4 · Run the engine live (short demo)

The full search ran offline on 20 CPU cores (the progress curve in the dashboard). Here we compile the **same C++ source** from the
attached dataset and give it a short budget on a sample of puzzles, starting from the best programs found in section 2, so you can
see it work and reuse it. Any program it shortens is replayed again before it enters the submission.
Set `DEMO_SECONDS = 0` to skip, or raise it (and `DEMO_PUZZLES`) to keep improving.

In [ ]:
DEMO_SECONDS, DEMO_PUZZLES, WORKERS = 20, 32, os.cpu_count() or 2
WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("working"); WORK.mkdir(exist_ok=True)
src = DS / "elfopt.cpp"; exe = WORK / "elfopt"
subprocess.run(["g++", "-std=c++17", "-O3", "-march=native", "-o", str(exe), str(src)], check=True)
print("compiled", exe, "|", sum(1 for _ in open(src)), "lines of C++")

def encode(p, prog):
    s = ["P", p.id, str(len(p.starts))] + [str(x) for x in p.starts] + [str(len(p.targets))] + [str(x) for x in p.targets]
    s += [str(p.limit), str(p.registers), str(len(prog))]
    return "\n".join([" ".join(s)] + ["%s %d %d %d" % tuple(x) for x in prog]) + "\n"
def parse_results(text):
    res, lines, k = {}, text.split("\n"), 0
    while k < len(lines):
        ln = lines[k].split()
        if ln and ln[0] == "R":
            pid, n = ln[1], int(ln[3])
            res[pid] = [[o, int(i), int(j), int(d)] for o, i, j, d in (lines[k + 1 + t].split() for t in range(n))]
            k += n + 1
        else: k += 1
    return res

demo_gain = 0
if DEMO_SECONDS > 0:
    rng = np.random.default_rng(2026)
    pool = [k for k in PUZ if FAM[k] in ("shared", "mixed", "multiplicative") and len(BEST[k]) >= 40]
    ids = list(rng.choice(pool, size=min(DEMO_PUZZLES, len(pool)), replace=False))
    shards = [ids[i::WORKERS] for i in range(WORKERS)]
    procs = []
    for i, sh in enumerate(shards):
        if not sh: continue
        (WORK / f"in_{i}.txt").write_text("".join(encode(PUZ[k], BEST[k]) for k in sh))
        procs.append(subprocess.Popen(f"{exe} --time {DEMO_SECONDS} --seed {i + 1} --kicks 1000 --kslack 2 < {WORK}/in_{i}.txt > {WORK}/out_{i}.txt 2> {WORK}/log_{i}.txt", shell=True))
    t0 = time.time()
    for pr in procs: pr.wait()
    for f in WORK.glob("out_*.txt"):
        for pid, prog in parse_results(f.read_text()).items():
            n = replay(PUZ[pid], prog)
            if n < len(BEST[pid]): demo_gain += len(BEST[pid]) - n; BEST[pid] = prog; WHO[pid] = "live demo"
    logs = "".join(f.read_text() for f in WORK.glob("log_*.txt"))
    print(f"demo: {len(ids)} puzzles x {DEMO_SECONDS}s on {WORKERS} workers in {time.time() - t0:.0f}s -> {demo_gain} instructions saved")
    print("\n".join(logs.strip().split("\n")[:12]))

## 5 · Anatomy of our rewrites

Each panel pair shows only the part of a program that changed. Left: the instructions the public program used (red) and the values
they fed. Right: our replacement (green). Grey boxes are values that already existed elsewhere in the program; blue boxes are targets.
The typical saving comes from **re-using a value that the program already had** — often another target — instead of building a
fresh chain.

In [ ]:
style()
picks = SHOW[1:4] or SHOW[:1]
if not picks: print("No puzzle is shorter than every public program in this version — nothing to draw.")
fig, axs = plt.subplots(max(len(picks), 1), 2, figsize=(17, 5.2 * max(len(picks), 1)), squeeze=False)
for r, k in enumerate(picks):
    d0, d1 = dag_of(PUZ[k], FRONTIER[k]), dag_of(PUZ[k], OURS[k])
    rewrite_panel(axs[r, 0], PUZ[k], d0, d1, "old", f"{k} · public ({len(FRONTIER[k])} instructions): removed")
    rewrite_panel(axs[r, 1], PUZ[k], d0, d1, "new", f"{k} · ours ({len(OURS[k])} instructions): replacement")
plt.tight_layout(); plt.show()

## 6 · The same puzzle, whole program

The full value DAG of the dashboard's example before and after, laid out by arithmetic depth from the starting values
(squares). Most of the graph is untouched; the search only rewires a small region.

In [ ]:
style()
k = SHOW[0]; d0, d1 = dag_of(PUZ[k], FRONTIER[k]), dag_of(PUZ[k], OURS[k])
fig, axs = plt.subplots(1, 2, figsize=(17, 6.2))
draw_dag(axs[0], PUZ[k], d0, frozenset(v for v in d0 if v not in d1), f"{k} · public · {len(FRONTIER[k])} instructions", hl_color=RED)
draw_dag(axs[1], PUZ[k], d1, frozenset(v for v in d1 if v not in d0), f"{k} · ours · {len(OURS[k])} instructions", hl_color=AQUA)
plt.tight_layout(); plt.show()

## 7 · Write the submission

`submission.csv` is **exactly our attached programs**, so the file's score is the score in the title and on the leaderboard chip
of this version (132,256). The per-puzzle best over every attached source and the live demo is written separately to
`submission_live_merge.csv` — if the attached public notebooks improved after we ran, that file can be lower; submit it if you want
the extra instructions. Both files are written, read back and replayed once more.

In [ ]:
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path(".")
def write_and_check(path, progs):
    with open(path, "w", newline="") as f:
        w = csv.writer(f); w.writerow(["id", "program"])
        for pid in PUZ:
            cell = json.dumps(progs[pid], separators=(",", ":")); assert len(cell.encode()) <= 131072
            w.writerow([pid, cell])
    check = read_submission(path)
    return sum(replay(PUZ[pid], check[pid]) for pid in PUZ)
total = write_and_check(WORK_DIR / "submission.csv", OURS)
live = write_and_check(WORK_DIR / "submission_live_merge.csv", BEST)
assert total == 132256, (total, 132256)
print(f"submission.csv            : 1000 programs, all valid, total = {total:,} instructions (= title score)")
print(f"submission_live_merge.csv : 1000 programs, all valid, total = {live:,} instructions (ours + live public + demo)")
for f in WORK_DIR.glob("*.txt"): f.unlink()          # keep the output tab clean (engine shards)

## What did not work (so you don't have to try)

* **Re-running the public value-DAG LNS on the frontier**: 0 shorter programs on 100 puzzles × 15 s. The frontier is a deep local
  optimum for depth-≤4 rebuilds of single cones.
* **A greedy constructive solver with our depth-5 search** (build the cheapest remaining target, split the rest as `X·Y ± R`):
  271 instructions on `scored_0001` vs 230 public, and 70 simultaneously live values (> 48 registers). Even with 270 values in the pool,
  no 50-bit target was reachable in ≤ 5 instructions — sharing helpers across targets matters more than per-target depth.
* **The joint pair search** (one new helper serving two targets) only occasionally beats sequential repair with two orders; it is
  cheap, so we keep it, but most gains come from single-cone and helper-rooted moves plus plateau walking.
* **A full depth-5 meet-in-the-middle** (`T = A op B` with `A, B` ranging over all ~10–19 million two-step values of the whole DAG pool):
  0 hits on 9 targets of 48–59 bits whose exclusive cones have 6–8 instructions. With `value_limit` just above the targets, two-step
  values are far too sparse at that magnitude — deeper per-target search is not where the slack is.
* **CP-SAT recombination, the variants that did not help:** with the order of the current program *fixed*, all 8 test puzzles were
  already optimal (only the free-order model with level variables finds gains). Adding “bridge” intermediates shared by ≥ 2 values,
  or enumerating every ≤ 3-instruction alternative build per target (300–900 values per puzzle), made the model too large to beat
  the hint in 30 s. The large *independent* puzzles (≥ 180 instructions) gained nothing from recombination or LNS.
* **Puzzle-family “hidden generator” chains** (`t_j = m·t_i + r` in multiplicative puzzles) — no simple chain that is cheaper.

## Reproduce / continue

`driver.py` in the dataset is the LNS loop: shard the current best file, run `elfopt` on every core with fresh seeds
(`--kicks 1000 --kslack 2`), replay every returned program, keep strictly shorter ones, repeat.
`recomb_pass.py` is one LNS-harvest + CP-SAT recombination pass (`elfopt --dump` writes the visited recipes, `recombine.py`
solves the free-order model per puzzle); `loop.sh` repeats it. We focus both on puzzles of 60–180 instructions: in our logs they
produced almost every gain (14 instructions per 100 runs for 100–140-instruction puzzles vs ≈0 above 180).

*Score history and every program are in the attached dataset. If this helped you, an upvote here and on the credited notebooks is
appreciated.*